# Window Functions

The recommendation engine needs rankings, running totals, and comparisons
with previous values. GROUP BY cannot do this -- it collapses rows into
summaries. We need a way to calculate something *across* a set of rows
while keeping every individual row intact.

That is exactly what window functions do.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
%pip install -q jupysql
%load_ext sql

In [ ]:
%sql sqlite:///../data/wavelength.sqlite

## The problem GROUP BY cannot solve

Suppose we want to rank tracks within each genre by play count. GROUP BY
would give us one row per genre -- but we want every track, with a rank
number next to it.

In [ ]:
%%sql
-- GROUP BY collapses: one row per genre
SELECT genre, MAX(play_count) AS top_play_count
FROM tracks
WHERE play_count IS NOT NULL
GROUP BY genre;

Six rows. One per genre. Window functions give us the ranking while keeping
all the individual tracks.

In [ ]:
%%sql
SELECT
    title,
    genre,
    play_count,
    RANK() OVER (PARTITION BY genre ORDER BY play_count DESC) AS genre_rank
FROM tracks
WHERE play_count IS NOT NULL
ORDER BY genre, genre_rank
LIMIT 20;

Every track is still there, but each one now has a `genre_rank`. Let's
break down the syntax:

- `RANK()` is the window function -- it assigns a rank number
- `OVER (...)` defines the "window" the function operates on
- `PARTITION BY genre` divides rows into groups (like GROUP BY, but without
  collapsing)
- `ORDER BY play_count DESC` determines ranking order within each partition

## ROW_NUMBER, RANK, and DENSE_RANK

These three functions all assign numbers to rows, but they handle ties
differently. This matters more than you might think.

In [ ]:
%%sql
SELECT
    title,
    genre,
    play_count,
    ROW_NUMBER() OVER (PARTITION BY genre ORDER BY play_count DESC) AS row_num,
    RANK() OVER (PARTITION BY genre ORDER BY play_count DESC) AS rank,
    DENSE_RANK() OVER (PARTITION BY genre ORDER BY play_count DESC) AS dense_rank
FROM tracks
WHERE play_count IS NOT NULL
  AND genre = 'rock'
ORDER BY play_count DESC
LIMIT 15;

Here is how they differ when there are ties:

- **ROW_NUMBER** always gives unique consecutive numbers. Ties are broken
  arbitrarily. No gaps.
- **RANK** gives the same number to ties, then skips. Two tracks tie for
  3rd: both get 3, next gets 5.
- **DENSE_RANK** gives the same number to ties without skipping. Two tracks
  tie for 3rd: both get 3, next gets 4.

Use ROW_NUMBER when you need exactly N results. Use RANK for competitive
rankings. Use DENSE_RANK when you want to know how many distinct ranking
levels exist.

## PARTITION BY vs no partition

If you omit PARTITION BY, the window function treats all rows as one big
group.

In [ ]:
%%sql
SELECT
    title,
    genre,
    play_count,
    RANK() OVER (ORDER BY play_count DESC) AS overall_rank,
    RANK() OVER (PARTITION BY genre ORDER BY play_count DESC) AS genre_rank
FROM tracks
WHERE play_count IS NOT NULL
ORDER BY play_count DESC
LIMIT 15;

A track might be ranked 50th overall but 1st in classical.

## LAG and LEAD: looking at neighbouring rows

This is where window functions become truly indispensable.

`LAG(column, n)` looks at the value from `n` rows *before* the current row.
`LEAD(column, n)` looks `n` rows *after*. The recommendation engine wants
to know: "For each user, how much time passed between consecutive listens?"
Without window functions, you would need a self-join. With LAG, it is one
line.

In [ ]:
%%sql
SELECT
    user_id,
    listened_at,
    LAG(listened_at, 1) OVER (
        PARTITION BY user_id
        ORDER BY listened_at
    ) AS previous_listen
FROM listens
WHERE user_id IN (1, 2)
ORDER BY user_id, listened_at
LIMIT 15;

Each row shows its own timestamp and the previous listen's timestamp within
the same user. The first listen for each user has NULL -- there is no prior
listen.

We can calculate the actual gap. SQLite does not have native datetime
arithmetic, but `julianday()` converts timestamps to numbers we can
subtract.

In [ ]:
%%sql
WITH listen_gaps AS (
    SELECT
        user_id,
        listened_at,
        LAG(listened_at, 1) OVER (
            PARTITION BY user_id
            ORDER BY listened_at
        ) AS previous_listen
    FROM listens
)
SELECT
    user_id,
    listened_at,
    previous_listen,
    ROUND(
        (julianday(listened_at) - julianday(previous_listen)) * 24, 1
    ) AS hours_since_previous
FROM listen_gaps
WHERE previous_listen IS NOT NULL
  AND user_id <= 3
ORDER BY user_id, listened_at
LIMIT 15;

In a single query, we see exactly how many hours passed between each pair
of listens. Try doing that with a self-join -- it would be three times as
long and half as readable. LAG and LEAD are the Swiss army knife of
analytics SQL.

## Running totals with SUM OVER

A running total adds up values as you move through the rows. This is
invaluable for tracking growth over time.

In [ ]:
%%sql
WITH daily_listens AS (
    SELECT
        DATE(listened_at) AS listen_date,
        COUNT(*) AS daily_count
    FROM listens
    GROUP BY DATE(listened_at)
)
SELECT
    listen_date,
    daily_count,
    SUM(daily_count) OVER (
        ORDER BY listen_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_listens
FROM daily_listens
ORDER BY listen_date
LIMIT 20;

The `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` means "sum
everything from the beginning up to this row". This is the default window
frame, so you can shorten it, but being explicit makes the intent clearer.

Add PARTITION BY and the running total resets for each group.

In [ ]:
%%sql
WITH daily_genre_listens AS (
    SELECT
        DATE(l.listened_at) AS listen_date,
        t.genre,
        COUNT(*) AS daily_count
    FROM listens l
    INNER JOIN tracks t ON l.track_id = t.track_id
    GROUP BY DATE(l.listened_at), t.genre
)
SELECT
    listen_date,
    genre,
    daily_count,
    SUM(daily_count) OVER (
        PARTITION BY genre
        ORDER BY listen_date
    ) AS cumulative_by_genre
FROM daily_genre_listens
WHERE genre IN ('rock', 'electronic')
ORDER BY genre, listen_date
LIMIT 20;

## Moving averages

You can use AVG as a window function too. A 3-day moving average smooths
out daily noise. The `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` creates a
sliding window of 3 rows.

In [ ]:
%%sql
WITH daily_listens AS (
    SELECT
        DATE(listened_at) AS listen_date,
        COUNT(*) AS daily_count
    FROM listens
    GROUP BY DATE(listened_at)
)
SELECT
    listen_date,
    daily_count,
    ROUND(AVG(daily_count) OVER (
        ORDER BY listen_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 1) AS three_day_avg
FROM daily_listens
ORDER BY listen_date
LIMIT 15;

## Top N per group

A classic analytics question: "What are the top 3 tracks in each genre?"
This combines ROW_NUMBER with a CTE. You cannot use a window function in
a WHERE clause directly -- it has not been calculated at that stage. The
CTE assigns ranks, and the outer query filters.

In [ ]:
%%sql
WITH ranked_tracks AS (
    SELECT
        t.title,
        a.name AS artist,
        t.genre,
        t.play_count,
        ROW_NUMBER() OVER (
            PARTITION BY t.genre
            ORDER BY t.play_count DESC
        ) AS genre_rank
    FROM tracks t
    INNER JOIN artists a ON t.artist_id = a.artist_id
    WHERE t.play_count IS NOT NULL
)
SELECT title, artist, genre, play_count, genre_rank
FROM ranked_tracks
WHERE genre_rank <= 3
ORDER BY genre, genre_rank;

This is a pattern you will use constantly.

## Percentage of total

An empty `OVER()` -- no partition, no order -- means "calculate across all
rows". This is perfect for computing percentages.

In [ ]:
%%sql
WITH genre_counts AS (
    SELECT
        t.genre,
        COUNT(*) AS listen_count
    FROM listens l
    INNER JOIN tracks t ON l.track_id = t.track_id
    GROUP BY t.genre
)
SELECT
    genre,
    listen_count,
    SUM(listen_count) OVER () AS total_listens,
    ROUND(100.0 * listen_count / SUM(listen_count) OVER (), 1) AS pct_of_total
FROM genre_counts
ORDER BY listen_count DESC;

## Putting it all together

Let's build a comprehensive artist dashboard: total listens, overall rank,
share of total listens, and comparison with the previous artist.

In [ ]:
%%sql
WITH artist_listens AS (
    SELECT
        a.name AS artist,
        a.genre,
        COUNT(*) AS total_listens
    FROM listens l
    INNER JOIN tracks t ON l.track_id = t.track_id
    INNER JOIN artists a ON t.artist_id = a.artist_id
    GROUP BY a.name, a.genre
)
SELECT
    artist,
    genre,
    total_listens,
    RANK() OVER (ORDER BY total_listens DESC) AS overall_rank,
    ROUND(100.0 * total_listens / SUM(total_listens) OVER (), 1) AS pct_share,
    LAG(total_listens, 1) OVER (ORDER BY total_listens DESC) AS prev_artist_listens,
    total_listens - LAG(total_listens, 1) OVER (ORDER BY total_listens DESC) AS gap_to_prev
FROM artist_listens
ORDER BY total_listens DESC
LIMIT 15;

That single query uses RANK, SUM OVER, and LAG -- three window functions
working together. The top artist shows NULL for `gap_to_prev` because
there is nobody above them.

## Exercises

### Exercise 1

Rank all tracks overall by play count (highest first) using DENSE_RANK.
Show the title, genre, play count, and rank. Limit to 20 rows.

### Exercise 2

For each user, show their listens with the previous listen timestamp using
LAG. Show user_id, listened_at, and previous_listen_at. Filter to users
1-5.

### Exercise 3

Calculate a running total of listens per user, ordered by timestamp. Show
user_id, listened_at, and cumulative_listens. Filter to user_id = 1.

### Exercise 4

Find the top 2 most-played tracks per genre using ROW_NUMBER and a CTE.
Show the title, genre, play count, and rank.

### Exercise 5

Calculate the percentage of total listens for each subscription type (free
vs premium) using SUM OVER. Show the subscription type, total listens, and
percentage.

### Exercise 6

Build a daily listens report with a 5-day moving average. Show the date,
daily count, and moving average (rounded to 1 decimal).

**Hint:** Use `ROWS BETWEEN 4 PRECEDING AND CURRENT ROW`.

## Summary

- Window functions calculate across rows **without collapsing them**
- `OVER()` defines the window: `PARTITION BY` groups rows, `ORDER BY`
  determines sequence
- **ROW_NUMBER** gives unique numbers; **RANK** handles ties with gaps;
  **DENSE_RANK** handles ties without gaps
- **LAG / LEAD** access neighbouring rows -- no self-join needed
- **SUM / AVG OVER** create running totals and moving averages
- An empty `OVER()` calculates across all rows -- useful for percentages
- The "top N per group" pattern uses ROW_NUMBER in a CTE, then filters

That concludes the SQL for Data Engineering module. You now have the
toolkit to answer the kinds of questions that product teams, analytics
teams, and recommendation engines ask every day. The key from here is
practice -- the more queries you write, the more natural the patterns
become.